# Demo 11 · Memory at scale

In part 3 today, ana asked for her locale and got bruno's. The store held two
people and one preference each, in a dict.

A real assistant holds thousands of memories per person, and finds the right one
by meaning, not by key. That is a vector search, and at scale it is almost always
**HNSW**: the index inside ChromaDB, Qdrant, pgvector and most of the others.

Three things to see, in about ten minutes:

1. **Fast search has a price.** HNSW answers quickly because it does not look at
   every memory, so it can miss the right one. You choose how often.
2. **A setting that says one thing and does another.** The same trap as part 3's
   preference the model never saw.
3. **Whose memory is it?** The owner stops being an argument nobody reads and
   becomes a filter on every search.

It runs on your laptop in seconds. No key, no account, no model download. It needs
ChromaDB, which `uv sync --extra projects --extra agents` installed in part 0.

In [ ]:
# manual-run: needs the `projects` extra (ChromaDB). Run it in class, after part 3.
import sys
import time
from pathlib import Path

try:
    import chromadb
    import numpy as np
except ImportError as error:
    raise SystemExit(
        f"missing {error.name!r}. Run: uv sync --extra projects --extra agents"
    ) from None

print(f"chromadb {chromadb.__version__}, numpy {np.__version__}")

## 1. Twenty thousand memories

An embedding model turns each memory into a list of numbers, a **vector**, so that
memories about the same thing end up close together. Here we make the vectors
directly instead, grouped into 200 topics. It skips a model download and changes
nothing the index sees: HNSW only ever works with the vectors.

In [ ]:
rng = np.random.default_rng(11)  # the same numbers every run, on every laptop
DIM, TOPICS, TOP_K = 128, 200, 10
centers = rng.normal(size=(TOPICS, DIM))


def make_memories(count):
    """`count` vectors, each near one of the topic centres, scaled to length 1."""
    vectors = centers[rng.integers(0, TOPICS, count)] + 0.9 * rng.normal(size=(count, DIM))
    vectors = vectors.astype(np.float32)
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


memories = make_memories(20_000)
questions = make_memories(50)  # 50 things someone could ask about
print(f"{len(memories):,} memories and {len(questions)} questions, {DIM} numbers each")

## 2. The exact answer, and what it costs

The exact way: compare the question with **every** memory and keep the ten
closest. It is always right. It also does more work for every memory you add.

In [ ]:
def exact_top_k(store, asked):
    """The true ten closest memories for each question: compare with all of them."""
    return np.argsort(-(asked @ store.T), axis=1)[:, :TOP_K]


start = time.perf_counter()
truth = exact_top_k(memories, questions)
exact_ms = (time.perf_counter() - start) / len(questions) * 1000
print(f"exact, 20,000 memories:  {exact_ms:.2f} ms per question")

bigger = make_memories(100_000)
start = time.perf_counter()
exact_top_k(bigger, questions)
print(f"exact, 100,000 memories: {(time.perf_counter() - start) / len(questions) * 1000:.2f} ms per question")

Five times the memories, about five times the time. Keep every conversation for a
year and the exact search slows down with it.

## 3. HNSW: a graph you walk, not a list you scan

HNSW links every memory to a few of its nearest neighbours, in layers: a sparse
top layer for long jumps and a dense bottom layer for the last steps. A search
starts at the top and walks towards the question, always moving to the closest
neighbour it can see. It never looks at most of the memories. That is why it is
fast, and why it can walk past the right one.

Three settings decide the trade. The names differ between stores; the ideas do not.

| What it controls | ChromaDB | Qdrant |
|---|---|---|
| Links per memory. More links, fewer misses, more RAM | `max_neighbors` | `m` |
| Effort while building the graph | `ef_construction` | `ef_construct` |
| Candidates kept while searching. More, fewer misses, slower | `ef_search` | `hnsw_ef` |

**Recall** is the number to watch: of the true ten closest memories, how many did
the search return? A memory the search misses is a memory your assistant never
sees, and nothing tells you.

In [ ]:
client = chromadb.EphemeralClient()  # in memory, gone when the notebook stops


def build(name, links, candidates):
    """A collection with its HNSW settings fixed at creation, holding all 20,000 memories."""
    collection = client.create_collection(
        name,
        configuration={"hnsw": {"space": "cosine", "max_neighbors": links, "ef_search": candidates}},
    )
    for first in range(0, len(memories), 5000):
        batch = range(first, min(first + 5000, len(memories)))
        collection.add(ids=[str(i) for i in batch], embeddings=memories[first : first + 5000])
    return collection


def measure(collection):
    """Recall against the exact answer, and milliseconds per question."""
    collection.query(query_embeddings=questions[:1], n_results=TOP_K)  # warm up
    start = time.perf_counter()
    found = collection.query(query_embeddings=questions, n_results=TOP_K)["ids"]
    ms = (time.perf_counter() - start) / len(questions) * 1000
    hits = [len({int(i) for i in ids} & set(truth[q])) / TOP_K for q, ids in enumerate(found)]
    return float(np.mean(hits)), ms


print("links  candidates   recall   ms/question")
for links in (4, 16):
    for candidates in (10, 50, 200):
        recall, ms = measure(build(f"links-{links}-ef-{candidates}", links, candidates))
        print(f"{links:>5}  {candidates:>10}   {recall:>6.2f}   {ms:>8.3f}")
print(f"\nexact search, for comparison: recall 1.00, {exact_ms:.3f} ms/question")

Read the table from the top:

- **4 links, 10 candidates** is the fastest, and finds only 3 or 4 of the true 10.
- **More candidates** buys recall back and costs time.
- **More links** needs fewer candidates for the same recall, and uses more memory.

Run it again and the numbers move a little: the graph is built with some
randomness, so the same settings give a slightly different index. The pattern does
not move.

None of these is "correct". You pick one by measuring recall on data like yours,
against the exact answer, the way this cell does. Not by copying a number from a
blog post.

## 4. The setting that says one thing and does another

Build a fast, sloppy index, then raise `ef_search` to 200 afterwards. Read what
the collection reports, then measure what it does.

In [ ]:
sloppy = build("sloppy", links=4, candidates=10)
recall_before, _ = measure(sloppy)

sloppy.modify(configuration={"hnsw": {"ef_search": 200}})
reported = client.get_collection("sloppy").configuration_json["hnsw"]["ef_search"]
recall_after, _ = measure(client.get_collection("sloppy"))

print(f"recall before the change: {recall_before:.2f}")
print(f"the collection reports:   ef_search = {reported}")
print(f"recall after the change:  {recall_after:.2f}")

The change was accepted and the configuration says 200. The recall did not move:
the index still searches with the 10 it was built with. Nothing raised.

That is part 3 again. A preference was stored and the prompt never carried it. Here
a setting is stored and the search never uses it. Both look done, and both are only
caught by checking the result instead of the thing you set.

This was measured on the ChromaDB version printed at the top. Another version may
behave differently, which is the point: **set index settings when you create the
collection, and measure recall to know what you got.**

## 5. Whose memory is it?

In part 3, `MemoryStore` took a `user_id` and never read it. In a vector store the
owner is **metadata** on each memory, and every search filters on it.

Bruno has 2,000 memories. Ana has five.

In [ ]:
store = client.create_collection("memories", configuration={"hnsw": {"space": "cosine"}})
store.add(
    ids=[f"bruno-{i}" for i in range(2000)],
    embeddings=memories[:2000],
    metadatas=[{"user_id": "bruno"}] * 2000,
)
store.add(
    ids=[f"ana-{i}" for i in range(5)],
    embeddings=memories[2000:2005],
    metadatas=[{"user_id": "ana"}] * 5,
)

ana_asks = questions[:1]

no_owner = store.query(query_embeddings=ana_asks, n_results=10)["ids"][0]
print("no filter:    ", sum(i.startswith("ana") for i in no_owner), "of 10 are ana's")

hers = store.query(query_embeddings=ana_asks, n_results=10, where={"user_id": "ana"})["ids"][0]
print("owner filter: ", hers)

Without the filter, ana gets bruno's memories, because they are closer to her
question. That is part 3's bug with a nicer name.

With the filter she asked for 10 and got 5: every memory she has. **Fewer than you
asked for is an answer, not an error.** Padding it with somebody else's memories
is the leak.

Notice what protected her: a `where=` on the query. That is a **filter**, the weaker
form from today's slides, because it is one line somebody forgets and the search
still works without it. The stronger form makes the owner part of the key, so a read
with no owner cannot be written at all: `(user_id, key)` in part 3, a namespace like
`("ana", "memories")` in LangGraph's store, or one collection per owner here.

The other two of today's four questions have a line each here:

In [ ]:
# How is it corrected? Replace a memory under the same id.
store.upsert(ids=["ana-0"], embeddings=memories[3000:3001], metadatas=[{"user_id": "ana"}])

# When does it stop being true? Delete it; it stops coming back at once.
store.delete(ids=["ana-1"])

left = store.query(query_embeddings=ana_asks, n_results=10, where={"user_id": "ana"})["ids"][0]
print("ana now:", sorted(left))

## For your capstone

Your capstone does not need a vector store. If it keeps memory in anything,
these hold:

- **Every read names the owner.** A search with no owner filter is part 3's bug,
  whatever the store is called.
- **Fewer results is a result.** Never fill the gap with somebody else's.
- **Measure the index, do not read its settings.** Compare against an exact search
  on a sample of your own data.
- **Write it down.** The owner, how a memory is corrected and when it expires go
  in `docs/RETENTION.md`, the five lines from part 5.

**Change one thing and run it again.** Try 32 links, or 50,000 memories, or delete
the `where=` from the last query and see whose memories come back.